##Forecasting on Graphs with GNNs using tsl library

In [1]:
!pip uninstall -y numpy pandas
!pip install numpy==1.26.4 pandas==2.2.2 --force-reinstall

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 517.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 18.1 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2025.2
    Uninstalling tzdata-2025.2:
      Successfully uninstalled tzdata-2025.2
 

In [1]:
# Install required packages.
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

# Install PyG dependencies
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

# Install tsl from source
!pip install -q git+https://github.com/TorchSpatiotemporal/tsl.git

2.6.0+cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 70.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 37.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.1/823.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.5/961.5 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
import tsl
import torch_geometric
import numpy as np
import pandas as pd

Datasets

In [3]:
from tsl import datasets
print("Datasets in tsl:", *datasets.dataset_classes, sep='\n- ')

Datasets in tsl:
- AirQuality
- Elergone
- EngRad
- MetrLA
- PemsBay
- PeMS03
- PeMS04
- PeMS07
- PeMS08
- LargeST
- PvUS
- ElectricityBenchmark
- TrafficBenchmark
- SolarBenchmark
- ExchangeBenchmark
- GaussianNoiseSyntheticDataset
- GPVARDataset
- GPVARDatasetAZ


Dataset loading

In [13]:
from tsl.datasets import GPVARDatasetAZ

dataset = GPVARDatasetAZ()

print(dataset)

Generating GPVARDatasetAZ data: 100%|██████████| 30000/30000 [00:14<00:00, 2059.28it/s]

GPVAR-AZ(length=30000, n_nodes=30, n_channels=1)


In [14]:
#print(f"Sampling period: {dataset.freq}")
print(f"Has missing values: {dataset.has_mask}")
print(f"Percentage of missing values: {(1 - dataset.mask.mean()) * 100:.2f}%")
print(f"Has exogenous variables: {dataset.has_covariates}")
print(f"Covariates: {', '.join(dataset.covariates.keys())}")

Has missing values: True
Percentage of missing values: 0.00%
Has exogenous variables: True
Covariates: optimal_pred


Connectivity

In [18]:
print(f"Default similarity: {dataset.similarity_score}")
print(f"Available similarity options: {dataset.similarity_options}")
print("==========================================")

#sim = dataset.get_similarity("distance")  # or dataset.compute_similarity()
#sim = dataset.compute_similarity("distance")

Default similarity: None
Available similarity options: None


In [19]:
connectivity = dataset.get_connectivity(threshold=0.1,
                                        include_self=False,
                                        normalize_axis=1,
                                        layout="edge_index")

In [20]:
edge_index, edge_weight = connectivity
from tsl.ops.connectivity import edge_index_to_adj
adj = edge_index_to_adj(edge_index, edge_weight)

PyTorch-ready dataset

In [21]:
from tsl.data import SpatioTemporalDataset

torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
                                      connectivity=connectivity,
                                      mask=dataset.mask,
                                      horizon=12,
                                      window=12,
                                      stride=1)
print(torch_dataset)

SpatioTemporalDataset(n_samples=29977, n_nodes=30, n_channels=1)


Preparing dataset for training

In [22]:
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler

# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=64,
)
dm.setup()
print(dm)

{Train dataloader: size=21572}
{Validation dataloader: size=2386}
{Test dataloader: size=5995}
{Predict dataloader: None}


Build Spatiotemporal Graph Neural Networks

In [23]:
import torch.nn as nn

from tsl.nn.blocks import RNN, MLPDecoder
from tsl.nn.layers import NodeEmbedding, DiffConv


class TimeThenSpaceModel(nn.Module):
    def __init__(self, input_size: int, n_nodes: int, horizon: int,
                 emb_size: int = 16,
                 hidden_size: int = 32,
                 rnn_layers: int = 1,
                 gnn_kernel: int = 2):
        super(TimeThenSpaceModel, self).__init__()

        self.node_embeddings = NodeEmbedding(n_nodes, emb_size)

        # Encoder
        self.encoder = nn.Linear(input_size + emb_size, hidden_size)

        # STMP
        self.time_nn = RNN(input_size=hidden_size,
                           hidden_size=hidden_size,
                           n_layers=rnn_layers,
                           cell='gru',
                           return_only_last_state=True)

        self.space_nn = DiffConv(in_channels=hidden_size,
                                 out_channels=hidden_size,
                                 k=gnn_kernel,
                                 root_weight=True)
        # Decoder
        self.decoder = MLPDecoder(input_size=hidden_size + emb_size,
                                  hidden_size=2 * hidden_size,
                                  output_size=input_size,
                                  horizon=horizon,
                                  n_layers=1)

    def forward(self, x, edge_index, edge_weight):
        # x: [batch time nodes features]
        b, t, n, f = x.size()
        # Concatenate node embeddings to input
        emb = self.node_embeddings(expand=(b, t, -1, -1))
        x_emb = torch.cat([x, emb], dim=-1)
        # Encoder
        x_enc = self.encoder(x_emb)  # linear proj: x_enc = [x||emb]Θ + b
        # STMP
        h = self.time_nn(x_enc)  # temporal processing: x=[b t n f] -> h=[b n f]
        z = self.space_nn(h, edge_index, edge_weight)  # spatial processing
        # Decoder
        emb = self.node_embeddings(expand=(b, -1, -1))
        z_emb = torch.cat([z, emb], dim=-1)  # concatenate node embeddings to z
        x_out = self.decoder(z_emb)  # linear proj: z=[b n f] -> x_out=[b n t⋅f]
        return x_out

In [28]:
import pytorch_lightning as pl
import torch
import torch.nn.functional as F

class Predictor(pl.LightningModule):
    def __init__(self, input_size, n_nodes, horizon,
                 emb_size=16, hidden_size=32, rnn_layers=1, gnn_kernel=2):
        super().__init__()
        self.model = TimeThenSpaceModel(input_size, n_nodes, horizon,
                                        emb_size, hidden_size, rnn_layers, gnn_kernel)
        self.loss_fn = torch.nn.L1Loss()  # MAE loss, because you're monitoring 'val_mae'

    def forward(self, x, edge_index, edge_weight=None): #modified edge_weight=None
        return self.model(x, edge_index, edge_weight)

    def training_step(self, batch, batch_idx):
        x, y, edge_index, edge_weight = batch
        y_hat = self(x, edge_index, edge_weight)
        loss = self.loss_fn(y_hat, y)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, edge_index, edge_weight = batch
        y_hat = self(x, edge_index, edge_weight)
        val_loss = self.loss_fn(y_hat, y)
        self.log('val_mae', val_loss, prog_bar=True)
        return val_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer


In [29]:
emb_size = 16      #@param
hidden_size = 32   #@param
rnn_layers = 1     #@param
gnn_kernel = 2     #@param

input_size = torch_dataset.n_channels   # 1 channel
n_nodes = torch_dataset.n_nodes         # 207 nodes
horizon = torch_dataset.horizon         # 12 time steps

stgnn = Predictor(input_size=input_size,
                           n_nodes=n_nodes,
                           horizon=horizon,
                           emb_size=emb_size,
                           hidden_size=hidden_size,
                           rnn_layers=rnn_layers,
                           gnn_kernel=gnn_kernel)
print(stgnn)

Predictor(
  (model): TimeThenSpaceModel(
    (node_embeddings): NodeEmbedding(n_nodes=30, embedding_size=16)
    (encoder): Linear(in_features=17, out_features=32, bias=True)
    (time_nn): RNN(
      (rnn): GRU(32, 32)
    )
    (space_nn): DiffConv(32, 32)
    (decoder): MLPDecoder(
      (readout): MLP(
        (mlp): Sequential(
          (0): Dense(
            (affinity): Linear(in_features=48, out_features=64, bias=True)
            (activation): ReLU()
            (dropout): Identity()
          )
        )
        (readout): Linear(in_features=64, out_features=12, bias=True)
      )
      (rearrange): Rearrange('b n (h f) -> b h n f', f=1, h=12)
    )
  )
  (loss_fn): L1Loss()
)


Training: predictor

In [30]:
from tsl.metrics.torch import MaskedMAE, MaskedMAPE
from tsl.engines import Predictor

from torchmetrics.metric import jit_distributed_available
from torchmetrics.utilities.data import dim_zero_sum
torch.serialization.add_safe_globals([
    MaskedMAE,
    jit_distributed_available,
    dim_zero_sum,
])

loss_fn = MaskedMAE()

metrics = {'mae': MaskedMAE(),
           'mape': MaskedMAPE(),
           'mae_at_15': MaskedMAE(at=2),  # '2' indicates the third time step,
                                          # which correspond to 15 minutes ahead
           'mae_at_30': MaskedMAE(at=5),
           'mae_at_60': MaskedMAE(at=11)}

# setup predictor
predictor = Predictor(
    model=stgnn,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 0.001},    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=metrics                # metrics to be logged during train/val/test
)

In [31]:
from pytorch_lightning.loggers import TensorBoardLogger

logger = TensorBoardLogger(save_dir="logs", name="tsl_intro", version=0)

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='logs',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
)

trainer = pl.Trainer(max_epochs=10, #set to 100
                     logger=logger,
                     accelerator="gpu" if torch.cuda.is_available() else "cpu",
                     devices=1,
                     limit_train_batches=10,  # set to 100
                     callbacks=[checkpoint_callback])

trainer.fit(predictor, datamodule=dm)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.callbacks.model_summary:
  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | loss_fn       | MaskedMAE        | 0      | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
3 | test_metrics  | MetricCollection | 0      | train
4 | model         | Predictor        | 16.5 K | train
-----------------------------------------------------------
16.5 K    Trainable params
0         Non-trainable params
16.5 K    Total params
0.066     Total estimated model params size (MB)
38        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (10) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


Testing

In [32]:
checkpoint = torch.load(checkpoint_callback.best_model_path, weights_only=False)

# Load weights into the predictor
predictor.load_state_dict(checkpoint['state_dict'])

# ---- TEST STEP ----
predictor.freeze()
trainer.test(predictor, datamodule=dm)

Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.6607198715209961     │
│         test_mae          │    0.6607199907302856     │
│      test_mae_at_15       │    0.5745146870613098     │
│      test_mae_at_30       │    0.6689742207527161     │
│      test_mae_at_60       │    0.7659023404121399     │
│         test_mape         │     2.381962299346924     │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 0.6607199907302856,
  'test_mae_at_15': 0.5745146870613098,
  'test_mae_at_30': 0.6689742207527161,
  'test_mae_at_60': 0.7659023404121399,
  'test_mape': 2.381962299346924,
  'test_loss': 0.6607198715209961}]

In [ ]:
#predictor.load_model(checkpoint_callback.best_model_path)
#predictor.freeze()

#trainer.test(predictor, datamodule=dm);